# product-slm: serving benchmark on a T4 (vLLM vs plain transformers)

This notebook measures what the serving layer actually delivers on one T4:
1. **vLLM with output constrained to the JSON schema.** This is the production path: base model + LoRA adapter, every output schema-valid by construction.
2. **vLLM unconstrained.** Does forcing the schema change accuracy or speed?
3. **Plain HF transformers, one title at a time.** The non-batched baseline on the same GPU.

Each run reports throughput and latency at batch sizes 1, 8, 32 and 150, plus accuracy on the 150 test titles. Accuracy is scored with the same metrics as training eval.

**Before you start:**
- Set **Runtime → Change runtime type → T4 GPU**.
- Have `qwen2.5-1.5b-lora-adapter.zip` ready (it's in `product-slm/outputs/` on your machine). Cell 2 asks you to upload it.

Then **Run all**. It takes ~15 min, mostly installing vLLM.

In [ ]:
import os
import subprocess
import sys


def sh(cmd: str) -> None:
    """Run a shell command with live output; raise if it fails."""
    proc = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed (exit {proc.returncode}): {cmd}")


print(sys.version)
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True
)
if gpu.returncode != 0:
    raise RuntimeError("No GPU: Runtime -> Change runtime type -> T4 GPU, then rerun.")
print(gpu.stdout)

In [ ]:
REPO = "/content/product-slm"
ADAPTER = f"{REPO}/outputs/qwen2.5-1.5b-lora/adapter"
if os.path.isdir(REPO):
    sh(f"git -C {REPO} pull -q")
else:
    sh(f"git clone -q https://github.com/houssem019/product-slm.git {REPO}")
os.chdir(REPO)

if not os.path.exists(f"{ADAPTER}/adapter_config.json"):
    try:
        from google.colab import files

        print("Upload qwen2.5-1.5b-lora-adapter.zip (from product-slm/outputs/ on your machine)")
        uploaded = files.upload()
        zip_name = next(iter(uploaded))
    except ImportError:  # Kaggle: add the zip as a dataset and set its path here
        zip_name = "/kaggle/input/product-slm-adapter/qwen2.5-1.5b-lora-adapter.zip"
    sh(f"mkdir -p {ADAPTER} && unzip -o -q '{zip_name}' -d {ADAPTER}")
assert os.path.exists(f"{ADAPTER}/adapter_config.json"), "adapter missing after upload"
sh(f"ls -la {ADAPTER}")

## Install
- **vLLM is pinned to v0.31.0**, the same version as `Dockerfile.gpu`. It brings its own torch/CUDA build.
- **`peft`** is for the plain-transformers baseline.
- **`torchao` is removed:** Colab's preinstalled copy is too old for peft (see `docs/CHALLENGES.md` #12).

In [ ]:
sh("pip install -q vllm==0.31.0 peft")
sh("pip install -q -e .")  # this repo + light deps (no torch extras, so vLLM's torch stays)
sh("pip uninstall -y -q torchao || true")
try:
    sh("vllm download-kernels")  # FlashInfer kernels; not needed by every vLLM build
except RuntimeError as e:
    print("note: kernel download skipped:", e)
sh(
    'python -c "import vllm, torch, transformers, peft; print(vllm.__version__, torch.__version__, transformers.__version__, peft.__version__)"'
)

## Benchmarks
Each run starts a fresh process, so GPU memory is released between runs.

In [ ]:
sh(f"python -m pslm.serve.bench --backend vllm --model {ADAPTER} --name vllm-guided")

In [ ]:
sh(f"python -m pslm.serve.bench --backend vllm --model {ADAPTER} --no-guided --name vllm-unguided")

In [ ]:
# Plain transformers, one title at a time: batch size doesn't change anything, so measure bs=1 only.
sh(f"python -m pslm.serve.bench --backend hf --model {ADAPTER} --batch-sizes 1 --name hf-t4")

## Results

In [ ]:
import json
from pathlib import Path

for name in ("vllm-guided", "vllm-unguided", "hf-t4"):
    b = json.loads(Path(f"data/bench/{name}.json").read_text())
    m = b["metrics"]
    print(f"== {name}  ({b['device']}, model load {b['model_load_s']} s)")
    print(
        f"   exact {m['exact_match']}  json_valid {m['json_valid']}  strict {b['strict_json']}  f1 {m['f1']}"
    )
    for r in b["runs"]:
        print(
            f"   bs={r['batch_size']:>3}: {r['titles_per_s']:>7} titles/s, p50 batch latency {r['batch_latency_ms_p50']} ms"
        )

In [ ]:
sh(
    "zip -qr serve_results.zip data/bench data/preds/test/qwen2.5-1.5b-lora-vllm-guided.* "
    "data/preds/test/qwen2.5-1.5b-lora-vllm-unguided.* data/preds/test/qwen2.5-1.5b-lora-hf-t4.*"
)
try:
    from google.colab import files

    files.download("serve_results.zip")
except ImportError:
    print("serve_results.zip written to", os.getcwd())